In [1]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://ko.wikipedia.org/wiki/인공지능",),
)
docs = loader.load()

print(f"불러온 문서 수: {len(docs)}")
print(f"글자 수: {len(docs[0].page_content)}")
print(docs[0].page_content[:500])

/var/folders/8b/tptjhw711rl8y2287cz1dzz00000gn/T/ipykernel_53594/3956788383.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


불러온 문서 수: 1
글자 수: 63246




인공지능 - 위키백과, 우리 모두의 백과사전



























본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















외관
















기부

계정 만들기

로그인








개인 도구






기부


계정 만들기


로그인





























목차
사이드바로 이동
숨기기




처음 위치





1
개념과 범위




개념과 범위 하위 문단 토글하기





1.1
인공지능과 인공지능 시스템








1.2
관련 개념의 관계








1.3
특화형 인공지능과 인공 일반 지능







In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits = text_splitter.split_documents(docs)

print(f"생성된 청크 수: {len(splits)}")
print(f"첫 번째 청크 미리보기:\n{splits[0].page_content[:300]}")

생성된 청크 수: 90
첫 번째 청크 미리보기:
인공지능 - 위키백과, 우리 모두의 백과사전



























본문으로 이동







주 메뉴





주 메뉴
사이드바로 이동
숨기기



		둘러보기
	


대문최근 바뀜요즘 화제임의 문서로





		사용자 모임
	


사랑방사용자 모임관리 요청





		편집 안내
	


소개도움말정책과 지침질문방



















검색











검색






















외관
















기부

계정 만들기

로그인








개인 도구



In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(),
)

print(f"저장된 청크 수: {vectorstore._collection.count()}")

저장된 청크 수: 90


In [4]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

question = "인공지능의 정의는 무엇인가?"
retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs):
    print(f"\n--- 청크 {i+1} ---")
    print(doc.page_content[:200])


--- 청크 1 ---
의식은 주관적인 경험이 있는 상태를 가리킨다. 사람과 동물의 의식은 행동, 신경계와 생물학적 구조를 함께 근거로 연구하지만, 인공지능에 어떤 증거를 적용해야 하는지는 합의되어 있지 않다. 언어 모델이 고통이나 감정을 표현하는 문장을 만든다는 사실만으로 실제 경험을 확인할 수는 없다.[86]
버틀린 등은 여러 의식 이론에서 인공지능에 적용할 수 있는 잠정적인

--- 청크 2 ---
인공지능은 반복적인 일을 줄이고 정보 접근성을 높이며 과학·의료·교육에 새로운 도구를 제공할 수 있다. 한편 잘못된 결과, 편향, 개인정보 침해, 허위 정보, 보안과 안전 문제, 노동과 환경에 미치는 영향도 낳을 수 있다. 성능과 위험은 과제와 사용 환경에 따라 달라지므로, 이를 평가하고 관리하기 위한 윤리적 논의와 법·제도적 규율도 함께 이루어지고 있다.

--- 청크 3 ---
인공지능은 문화유산의 보존과 접근성 향상에도 사용될 수 있다. 훼손된 사진·음원·문서를 복원하고, 손으로 쓴 기록을 문자로 바꾸며, 박물관과 기록관의 자료를 분류하거나 여러 언어로 설명할 수 있다. 유네스코는 인공지능이 유물과 기록의 디지털화·복원, 역사 자료 분석과 문화유산의 소개에 활용될 수 있다고 설명한다.[61]
생성형 인공지능과 저작권에는 서로 다

--- 청크 4 ---
개념
인공지능과의 관계

알고리즘
문제를 푸는 계산 절차이다. 모든 인공지능은 알고리즘을 사용하지만, 모든 알고리즘이 인공지능인 것은 아니다.

자동화
정해진 작업을 기계나 소프트웨어가 수행하게 하는 것이다. 인공지능을 사용하지 않고도 자동화할 수 있다.

기계 학습
여러 데이터 사례에서 규칙성을 찾아 모델을 조정하는 인공지능의 주요 방법이다.

딥 러닝



In [5]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

llm = ChatOpenAI(model="gpt-4o-mini")

prompt = ChatPromptTemplate.from_template("""
다음 컨텍스트만 참고해서 질문에 답변하세요. 컨텍스트에 없는 내용이면 모른다고 답하세요.

컨텍스트:
{context}

질문: {question}

답변:
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

context = format_docs(retrieved_docs)
chain = prompt | llm

response = chain.invoke({"context": context, "question": question})
print(response.content)

인공지능은 연구 분야나 기술을 가리키기도 하고, 그 기술로 만든 구체적인 시스템을 가리키기도 합니다. 경제협력개발기구는 인공지능 시스템을 목표를 위해 입력을 처리하여 예측, 생성물, 추천, 결정 등의 결과를 만들고 물리적 또는 가상 환경에 영향을 줄 수 있는 기계 기반 시스템으로 정의합니다. 시스템마다 자율성의 정도가 다르며, 일부는 실제 사용을 시작한 뒤 새로운 자료에 맞추어 동작을 조정할 수 있습니다.
